# Idealista API: raw -> preprocess consolidado

Este cuaderno consolida los CSV ya generados en preprocess.

Objetivo:
- leer todas las carpetas preprocess de una operacion (`rent` o `sale`)
- unir todos los CSV encontrados dentro de cada run
- eliminar duplicados priorizando los runs mas recientes
- exportar un unico CSV a `data/raw/idealistaAPI/preprocess/total_*_cantabria.csv`
- mostrar estadisticas por run: totales, unicos y duplicados


In [65]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "src").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("No se encontro la raiz del proyecto (carpeta 'src').")

PREPROCESS_BASE = PROJECT_ROOT / "data" / "raw" / "idealistaAPI" / "preprocess"


## Operacion a consolidar

Cambia `OPERATION` a `"sale"` o `"rent"`.


In [66]:
OPERATION = "rent"

OUTPUT_NAME_MAP = {
    "sale": "total_sales_cantabria.csv",
    "rent": "total_rent_cantabria.csv",
}

SUMMARY_NAME_MAP = {
    "sale": "total_sales_cantabria_summary.json",
    "rent": "total_rent_cantabria_summary.json",
}

if OPERATION not in OUTPUT_NAME_MAP:
    raise ValueError(f"Operacion no soportada: {OPERATION}")

OUTPUT_CSV = PREPROCESS_BASE / OUTPUT_NAME_MAP[OPERATION]
OUTPUT_SUMMARY = PREPROCESS_BASE / SUMMARY_NAME_MAP[OPERATION]

print(f"Operacion seleccionada: {OPERATION}")
print(f"OUTPUT_CSV: {OUTPUT_CSV}")


Operacion seleccionada: rent
OUTPUT_CSV: /home/pablo/Documents/bezanillasl/data/raw/idealistaAPI/preprocess/total_rent_cantabria.csv


## Runs preprocess detectados para la operacion


In [67]:
operation_runs = sorted(
    [path for path in PREPROCESS_BASE.glob(f"{OPERATION}_homes_run_*") if path.is_dir()],
    reverse=True,
)

if not operation_runs:
    raise FileNotFoundError(f"No se encontraron runs preprocess para operation='{OPERATION}' en {PREPROCESS_BASE}")

runs_df = pd.DataFrame(
    [
        {
            "run_name": path.name,
            "csv_files": len(list(path.glob("*.csv"))),
            "has_summary": (path / "summary.json").exists(),
        }
        for path in operation_runs
    ]
)

runs_df


,run_name,csv_files,has_summary
0,rent_homes_run_20260405_140420,1,True
1,rent_homes_run_20260401_135939,1,True
2,rent_homes_run_20260310_171627,1,True
3,rent_homes_run_20260220_111903,1,True


## Funciones de carga y deduplicado


In [68]:
def csv_name_for_operation(operation: str) -> str:
    return f"{operation}_homes_cantabria_bezana_like_raw.csv"


def build_property_dedupe_key(df: pd.DataFrame) -> pd.Series:
    property_code = df.get("propertyCode", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str).str.strip()
    fallback = (
        df.get("price", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str)
        + "|"
        + df.get("size", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str)
        + "|"
        + df.get("latitude", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str)
        + "|"
        + df.get("longitude", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str)
        + "|"
        + df.get("address", df.get("streetName", pd.Series([""] * len(df), index=df.index))).fillna("").astype(str).str.strip().str.lower()
    )
    return property_code.where(property_code != "", fallback)


def load_operation_runs_as_dataframe(run_dirs: list[Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames: list[pd.DataFrame] = []
    run_stats: list[dict] = []
    expected_csv_name = csv_name_for_operation(OPERATION)

    for run_dir in run_dirs:
        csv_path = run_dir / expected_csv_name
        if not csv_path.exists():
            print(f"{run_dir.name}: csv no encontrado ({expected_csv_name})")
            run_stats.append(
                {
                    "run_name": run_dir.name,
                    "csv_path": str(csv_path),
                    "rows_total": 0,
                    "rows_unique": 0,
                    "rows_duplicated": 0,
                    "missing_csv": True,
                }
            )
            continue

        df_run = pd.read_csv(csv_path)
        df_run["_dedupe_key"] = build_property_dedupe_key(df_run)

        rows_total = int(len(df_run))
        rows_duplicated = int(df_run.duplicated(subset="_dedupe_key").sum())
        rows_unique = rows_total - rows_duplicated

        print(f"{run_dir.name}: total={rows_total} unicos={rows_unique} duplicados={rows_duplicated}")

        run_stats.append(
            {
                "run_name": run_dir.name,
                "csv_path": str(csv_path),
                "rows_total": rows_total,
                "rows_unique": rows_unique,
                "rows_duplicated": rows_duplicated,
                "missing_csv": False,
            }
        )

        df_run = df_run.drop(columns=["_dedupe_key"])
        df_run["source_run"] = run_dir.name
        frames.append(df_run)

    if not frames:
        raise ValueError(f"No se encontraron CSV validos para operation='{OPERATION}'")

    combined_df = pd.concat(frames, ignore_index=True, sort=False)
    run_stats_df = pd.DataFrame(run_stats)
    return combined_df, run_stats_df


## Carga consolidada de la operacion


In [69]:
df_raw, run_stats_df = load_operation_runs_as_dataframe(operation_runs)

print(f"Runs incluidos: {len(operation_runs)}")
print(f"Filas raw totales: {len(df_raw)}")
print(f"Columnas: {len(df_raw.columns)}")

df_raw.head()


rent_homes_run_20260405_140420: total=486 unicos=486 duplicados=0
rent_homes_run_20260401_135939: total=488 unicos=488 duplicados=0
rent_homes_run_20260310_171627: total=524 unicos=524 duplicados=0
rent_homes_run_20260220_111903: total=2365 unicos=493 duplicados=1872
Runs incluidos: 4
Filas raw totales: 3863
Columnas: 49


,propertyCode,thumbnail,externalReference,numPhotos,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,address,province,municipality,district,country,latitude,longitude,showAddress,url,distance,description,hasVideo,status,newDevelopment,hasLift,priceByArea,hasPlan,has3DTour,has360,hasStaging,notes,topNewDevelopment,newDevelopmentHighlight,topPlus,priceInfo.price.amount,priceInfo.price.currencySuffix,detailedType.typology,suggestedTexts.subtitle,suggestedTexts.title,parkingSpace.hasParkingSpace,parkingSpace.isParkingSpaceIncludedInPrice,detailedType.subTypology,parkingSpace.parkingSpacePrice,priceInfo.price.priceDropInfo.formerPrice,priceInfo.price.priceDropInfo.priceDropValue,priceInfo.price.priceDropInfo.priceDropPercentage,source_run
0,111081830,https://img4.idealista.com/blur/480_360_mq/0/i...,zzm90759,18,2,900.0,flat,rent,100.0,True,3,2,calle Rucandial,Cantabria,Santander,Peñacastillo - Nuevamontaña,es,43.458078,-3.870885,False,https://www.idealista.com/inmueble/111081830/,4980,Zazume te trae este DÚPLEX con BALCÓN y APARCA...,False,good,False,False,9.0,False,False,False,False,[],False,False,False,900.0,€/mes,flat,"Peñacastillo - Nuevamontaña, Santander",Piso en calle Rucandial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rent_homes_run_20260405_140420
1,111078427,https://img4.idealista.com/blur/480_360_mq/0/i...,ARS2775i,28,5,980.0,flat,rent,85.0,True,3,2,calle Gertrudis Gómez,Cantabria,Santander,Peñacastillo - Nuevamontaña,es,43.439190,-3.856840,False,https://www.idealista.com/inmueble/111078427/,2699,¡Descubre este maravilloso piso de 85 metros c...,False,good,False,True,12.0,False,False,False,False,[],False,False,False,980.0,€/mes,flat,"Peñacastillo - Nuevamontaña, Santander",Piso en calle Gertrudis Gómez,True,True,NaN,NaN,NaN,NaN,NaN,rent_homes_run_20260405_140420
2,111060573,https://img4.idealista.com/blur/480_360_mq/0/i...,86135,20,4,975.0,flat,rent,81.0,True,3,2,Peñacastillo - Nuevamontaña,Cantabria,Santander,Peñacastillo - Nuevamontaña,es,43.439995,-3.856238,False,https://www.idealista.com/inmueble/111060573/,2785,¿Buscas un buen piso de alquiler zona Peñacast...,False,good,False,True,12.0,False,False,False,False,[],False,False,False,975.0,€/mes,flat,"Peñacastillo - Nuevamontaña, Santander",Piso,True,True,NaN,NaN,NaN,NaN,NaN,rent_homes_run_20260405_140420
3,111051080,https://img4.idealista.com/blur/480_360_mq/0/i...,NaN,17,1,750.0,flat,rent,40.0,True,1,1,calle Lavapiés,Cantabria,Santander,Alisal - Cazoña - San Román,es,43.464611,-3.841902,False,https://www.idealista.com/inmueble/111051080/,5602,"////A Sólo 1,4 kilometros del hospital de Vald...",True,good,False,True,19.0,False,False,False,False,[],False,False,False,750.0,€/mes,flat,"Alisal - Cazoña - San Román, Santander",Piso en calle Lavapiés,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rent_homes_run_20260405_140420
4,111051186,https://img4.idealista.com/blur/480_360_mq/0/i...,2903-80021-VARGAS,7,1,800.0,flat,rent,65.0,True,2,1,calle Vargas,Cantabria,Santander,Numancia - San Fernando,es,43.459966,-3.818296,False,https://www.idealista.com/inmueble/111051186/,5771,"VARGAS (Arco Iris) piso con terraza, junto a l...",False,good,False,True,12.0,False,False,False,False,[],False,False,False,800.0,€/mes,flat,"Numancia - San Fernando, Santander",Piso en calle Vargas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rent_homes_run_20260405_140420


## Resumen por run


In [70]:
run_stats_df


,run_name,csv_path,rows_total,rows_unique,rows_duplicated,missing_csv
0,rent_homes_run_20260405_140420,/home/pablo/Documents/bezanillasl/data/raw/ide...,486,486,0,False
1,rent_homes_run_20260401_135939,/home/pablo/Documents/bezanillasl/data/raw/ide...,488,488,0,False
2,rent_homes_run_20260310_171627,/home/pablo/Documents/bezanillasl/data/raw/ide...,524,524,0,False
3,rent_homes_run_20260220_111903,/home/pablo/Documents/bezanillasl/data/raw/ide...,2365,493,1872,False


## Deduplicado consolidado


In [71]:
df_raw["_dedupe_key"] = build_property_dedupe_key(df_raw)
duplicate_rows = int(df_raw.duplicated(subset="_dedupe_key").sum())
df_preprocess = df_raw.drop_duplicates(subset="_dedupe_key", keep="first").drop(columns=["_dedupe_key"]).copy()

print(f"Filas raw totales: {len(df_raw)}")
print(f"Duplicados detectados en consolidacion: {duplicate_rows}")
print(f"Filas preprocess finales: {len(df_preprocess)}")

df_preprocess.head()


Filas raw totales: 3863
Duplicados detectados en consolidacion: 2988
Filas preprocess finales: 875


,propertyCode,thumbnail,externalReference,numPhotos,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,address,province,municipality,district,country,latitude,longitude,showAddress,url,distance,description,hasVideo,status,newDevelopment,hasLift,priceByArea,hasPlan,has3DTour,has360,hasStaging,notes,topNewDevelopment,newDevelopmentHighlight,topPlus,priceInfo.price.amount,priceInfo.price.currencySuffix,detailedType.typology,suggestedTexts.subtitle,suggestedTexts.title,parkingSpace.hasParkingSpace,parkingSpace.isParkingSpaceIncludedInPrice,detailedType.subTypology,parkingSpace.parkingSpacePrice,priceInfo.price.priceDropInfo.formerPrice,priceInfo.price.priceDropInfo.priceDropValue,priceInfo.price.priceDropInfo.priceDropPercentage,source_run
0,111081830,https://img4.idealista.com/blur/480_360_mq/0/i...,zzm90759,18,2,900.0,flat,rent,100.0,True,3,2,calle Rucandial,Cantabria,Santander,Peñacastillo - Nuevamontaña,es,43.458078,-3.870885,False,https://www.idealista.com/inmueble/111081830/,4980,Zazume te trae este DÚPLEX con BALCÓN y APARCA...,False,good,False,False,9.0,False,False,False,False,[],False,False,False,900.0,€/mes,flat,"Peñacastillo - Nuevamontaña, Santander",Piso en calle Rucandial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rent_homes_run_20260405_140420
1,111078427,https://img4.idealista.com/blur/480_360_mq/0/i...,ARS2775i,28,5,980.0,flat,rent,85.0,True,3,2,calle Gertrudis Gómez,Cantabria,Santander,Peñacastillo - Nuevamontaña,es,43.439190,-3.856840,False,https://www.idealista.com/inmueble/111078427/,2699,¡Descubre este maravilloso piso de 85 metros c...,False,good,False,True,12.0,False,False,False,False,[],False,False,False,980.0,€/mes,flat,"Peñacastillo - Nuevamontaña, Santander",Piso en calle Gertrudis Gómez,True,True,NaN,NaN,NaN,NaN,NaN,rent_homes_run_20260405_140420
2,111060573,https://img4.idealista.com/blur/480_360_mq/0/i...,86135,20,4,975.0,flat,rent,81.0,True,3,2,Peñacastillo - Nuevamontaña,Cantabria,Santander,Peñacastillo - Nuevamontaña,es,43.439995,-3.856238,False,https://www.idealista.com/inmueble/111060573/,2785,¿Buscas un buen piso de alquiler zona Peñacast...,False,good,False,True,12.0,False,False,False,False,[],False,False,False,975.0,€/mes,flat,"Peñacastillo - Nuevamontaña, Santander",Piso,True,True,NaN,NaN,NaN,NaN,NaN,rent_homes_run_20260405_140420
3,111051080,https://img4.idealista.com/blur/480_360_mq/0/i...,NaN,17,1,750.0,flat,rent,40.0,True,1,1,calle Lavapiés,Cantabria,Santander,Alisal - Cazoña - San Román,es,43.464611,-3.841902,False,https://www.idealista.com/inmueble/111051080/,5602,"////A Sólo 1,4 kilometros del hospital de Vald...",True,good,False,True,19.0,False,False,False,False,[],False,False,False,750.0,€/mes,flat,"Alisal - Cazoña - San Román, Santander",Piso en calle Lavapiés,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rent_homes_run_20260405_140420
4,111051186,https://img4.idealista.com/blur/480_360_mq/0/i...,2903-80021-VARGAS,7,1,800.0,flat,rent,65.0,True,2,1,calle Vargas,Cantabria,Santander,Numancia - San Fernando,es,43.459966,-3.818296,False,https://www.idealista.com/inmueble/111051186/,5771,"VARGAS (Arco Iris) piso con terraza, junto a l...",False,good,False,True,12.0,False,False,False,False,[],False,False,False,800.0,€/mes,flat,"Numancia - San Fernando, Santander",Piso en calle Vargas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rent_homes_run_20260405_140420


## Export del CSV total


In [72]:
PREPROCESS_BASE.mkdir(parents=True, exist_ok=True)
df_preprocess.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

summary = {
    "operation": OPERATION,
    "preprocess_base": str(PREPROCESS_BASE),
    "runs_included": [run_dir.name for run_dir in operation_runs],
    "output_csv": str(OUTPUT_CSV),
    "rows_raw": int(len(df_raw)),
    "rows_output": int(len(df_preprocess)),
    "duplicates_removed": int(duplicate_rows),
    "priority_rule": "latest_run_first",
    "run_stats": run_stats_df.to_dict(orient="records"),
}

OUTPUT_SUMMARY.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"CSV exportado en: {OUTPUT_CSV.resolve()}")
summary


CSV exportado en: /home/pablo/Documents/bezanillasl/data/raw/idealistaAPI/preprocess/total_rent_cantabria.csv


{'operation': 'rent',
 'preprocess_base': '/home/pablo/Documents/bezanillasl/data/raw/idealistaAPI/preprocess',
 'runs_included': ['rent_homes_run_20260405_140420',
  'rent_homes_run_20260401_135939',
  'rent_homes_run_20260310_171627',
  'rent_homes_run_20260220_111903'],
 'output_csv': '/home/pablo/Documents/bezanillasl/data/raw/idealistaAPI/preprocess/total_rent_cantabria.csv',
 'rows_raw': 3863,
 'rows_output': 875,
 'duplicates_removed': 2988,
 'priority_rule': 'latest_run_first',
 'run_stats': [{'run_name': 'rent_homes_run_20260405_140420',
   'csv_path': '/home/pablo/Documents/bezanillasl/data/raw/idealistaAPI/preprocess/rent_homes_run_20260405_140420/rent_homes_cantabria_bezana_like_raw.csv',
   'rows_total': 486,
   'rows_unique': 486,
   'rows_duplicated': 0,
   'missing_csv': False},
  {'run_name': 'rent_homes_run_20260401_135939',
   'csv_path': '/home/pablo/Documents/bezanillasl/data/raw/idealistaAPI/preprocess/rent_homes_run_20260401_135939/rent_homes_cantabria_bezana_lik